# Análisis exploratorio de datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression

El **análisis exploratorio de datos**, también conocido como EDA (*Exploratory Data Analysis*) es el primer paso para resolver cualquier problema de Machine Learning.

Consiste en un proceso que busca analizar e investigar los conjuntos de datos de los que disponen y resumir sus principales características, empleando a menudo técnicas de visualización de datos.

Este análisis se lleva a cabo a través de una serie de pasos que se detallan a continuación.

1. Definición del problema
2. Recopilación de datos
3. Análisis Descriptivo
4. Limpieza de datos
5. Análisis de variables (univariante, multivariante)
6. Ingeniería de características (Duplicados, Faltantes, Nuevas características)
7. Split
8. Scaling & Encoding
9. Selección de características

Luego de la implementación y adopción de estos pasos, estaremos preparados para entrenar el modelo de Machine Learning.

## Paso 1: Definición del problema


Se cuenta con un conjunto de datos vinculados a los precios de los productos que se distribuyen en Uruguay, informados SIPC (Sistema de Información de Precios al Consumidor). Se desea preparar los datos para entrenar un modelo de machine learning que pueda asistir en la compra de productos y/o canasta familiar, contemplando los siguientes atributos:

- Tipo de producto
- Época del año
- Cadena

Se espera que el modelo pueda responder a los siguientes planteamientos:

- Precio promedio de los productos
- Cadenas que ofrecen un mejor costo del producto
- Sugerencia de productos por rango de precios

## Paso 2: Recopilación de datos
> Dataset de precios.uy

Observamos información del dataset (Precios):

| Variable | Definition | Key |
|:---:|:---|:---|
| `Periodo` | Mes y año del relevamiento de precios | Ejemplo: `Ene-25`, `Feb-25`, `Abr-25` |
| `Grupo` | Categoría general a la que pertenece el producto | Ejemplo: `Alimentos y bebidas`, `Cuidado personal`, `Limpieza del hogar`, `Frutas y verduras` |
| `Producto` | Nombre o descripción del producto relevado | Ejemplo: aceite, yerba, champú, pañales, etc. |
| `Super` | Supermercado o cadena donde se relevó el precio | Ejemplo: `Devoto`, `Disco`, `Ta - Ta`, `Tienda Inglesa`, etc. |
| `Precio` | Precio registrado para el producto en ese supermercado y período | Valor numérico expresado en pesos |


### Importamos los datos y creamos el DataFrame

In [ ]:
df = pd.read_csv('../data/raw/p4ds_cadenas_unificadas_2025.csv', sep=';')

In [ ]:
df.head()

## Paso 3: Análisis Descriptivo

Una vez que hemos cargado el conjunto de datos, debemos analizarlo en su totalidad, sin distinción de train y test, para obtener conclusiones conjuntas. Una vez que tenemos la información cargada en una estructura de datos manejable como es un DataFrame de Pandas, podemos arrancar con el proceso.

In [ ]:
# formateo para que pandas muestre todas las columnas del dataframe y dos decimales
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
# Tamaño y columnas
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.columns

In [ ]:
# Tipos de datos y nulos
df.info()

In [ ]:
# Resumen estadístico
df.describe(include='all')

In [ ]:
# Valores nulos
nulos = df.isna().sum().sort_values(ascending=False)
nulos[nulos > 0]

In [ ]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())

df.duplicated()

In [ ]:
# Cantidad de valores únicos por columna
df.nunique().sort_values(ascending=False)

In [ ]:
#todas las filas que tienen al menos un valor nulo.
df[df.isnull().any(axis=1)]

In [ ]:
#cuántas filas tienen al menos un valor nulo.
df.isnull().any(axis=1).sum()

In [ ]:
# cuántos valores nulos hay en cada columna.
df.isnull().sum()

In [ ]:
# creo df para almacenar filas con al menos un valor nulo
df_nulos = df[df.isnull().any(axis=1)]
df_nulos

In [ ]:
#df = df.dropna()
#df.isnull().sum()
# Podría dropear nulos pero lo voy a hacer más adelante

In [ ]:
df.info()

df

In [ ]:
df.describe().T

### Observaciones:

Una vez revisada la información del dataframe, se pueden extraer las siguientes conclusiones:

- El dataset contiene un total de **26.834 filas** y **5 columnas**: `Periodo`, `Grupo`, `Producto`, `Super` y `Precio`.
- La variable `Periodo` no tiene valores nulos, por lo que todas las filas cuentan con un período asociado.
- La variable con mayor cantidad de valores nulos es `Super`, con **2.597 valores faltantes**.
- La variable `Precio` tiene **433 valores nulos**, por lo que hay productos/registros sin precio informado.
- La variable `Grupo` tiene **163 valores nulos** y `Producto` tiene **22 valores nulos**.
- En total, existen **3.191 filas con al menos un valor nulo**.
- Si se eliminan todas las filas con valores nulos usando `df.dropna()`, el dataset queda con **23.643 filas**.
- Todas las columnas aparecen inicialmente como categóricas/texto (`object` o `str`), incluso `Precio`.
- La columna `Precio` debería transformarse a numérica, ya que actualmente usa coma decimal, por ejemplo `80,00`.
- El dataset contiene información de **12 períodos**, **10 grupos**, **427 productos** y **17 supermercados** distintos.


## Paso 4: Limpieza de Datos

### Limpieza de datos: Eliminar duplicados

Un punto muy importante a tener en cuenta en este paso es eliminar aquellas instancias que pudieran estar duplicadas en el conjunto de datos. Esto es crucial debido a que, de dejarlos, el mismo punto tendría varias representaciones, lo cual es matemáticamente incoherente e incorrecto. Para ello, hemos de ser inteligentes buscando duplicados y conocer previamente si los hay y dónde, antes de eliminarlos. Además, tenemos que tener en cuenta que una instancia puede estar repetida independientemente del identificador que pueda tener, así que en este caso nos interesa eliminar del análisis la variable `PassengerId`, ya que podría estar mal generada.

In [ ]:
df.duplicated()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.duplicated().sum()

### Observaciones
>
> En este dataset se encontraron **1.625 filas duplicadas exactas**, es decir, registros donde todas las columnas tienen los mismos valores.
>
> Para evitar que estos registros repetidos afecten el análisis, se aplica la función `drop_duplicates()`.
>
> Luego de eliminar los duplicados, el dataset pasa de **26.834 filas** a **25.209 filas**.

### Limpieza de datos: Normalizacion de columnas

In [ ]:
columnas_texto = ["Periodo", "Grupo", "Producto", "Super"]

for col in columnas_texto:
    df[col] = df[col].str.strip()

df["Grupo"] = df["Grupo"].str.lower().str.capitalize()
df["Producto"] = df["Producto"].str.lower()
df["Super"] = df["Super"].str.title()

df.nunique()

In [ ]:
df["Periodo"] = (
    df["Periodo"]
    .str.strip()
    .str.replace("-", "", regex=False)
)

In [ ]:
# Normalización del precio
df["Precio"] = pd.to_numeric(
    df["Precio"].astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

### Observaciones
>
> En esta etapa se normalizaron las columnas de texto y se corrigió el formato de la columna `Precio` para facilitar el análisis posterior.
>
> - Se eliminaron espacios al inicio y al final de las columnas `Periodo`, `Grupo`, `Producto` y `Super` usando `.str.strip()`.
> - La columna `Grupo` se transformó a minúsculas y luego se capitalizó, para unificar valores como `Alimentos y bebidas` y `alimentos y bebidas`.
> - La columna `Super` se transformó con `.str.title()`, para estandarizar los nombres de supermercados.
> - La columna `Producto` se transformó a minúsculas, ya que tiene muchos valores distintos y conviene evitar diferencias por mayúsculas/minúsculas.
> - La columna `Precio` se convirtió de texto a número decimal (`float`), reemplazando la coma decimal por punto decimal.
> - También se eliminaron posibles puntos usados como separadores de miles en `Precio`, para evitar errores al convertir a número.
> - Esta limpieza permite que los conteos de categorías sean más correctos y que `Precio` pueda usarse en cálculos estadísticos, gráficos, análisis de outliers y comparaciones numéricas.
> - Luego de la normalización, se verificaron los valores únicos y las frecuencias con `df.nunique()`, `df["Grupo"].value_counts()` y `df["Super"].value_counts()`.

### Limpieza de datos: Eliminar información irrelevante

Cuando queremos preparar los datos para entrenar un modelo predictivo debemos responder a la siguiente pregunta:

- ¿Son todas las características imprescindibles para realizar una predicción?

Normalmente, esa pregunta es un rotundo no. Tenemos que ser objetivos y llevar a cabo este proceso previo antes de la fase de selección de características. Por lo tanto, aquí lo que trataremos de hacer es una eliminación controlada de aquellas variables que estamos seguros de que el algoritmo no va a utilizarlas en el proceso predictivo.

In [ ]:
df["Grupo"].value_counts()

In [ ]:
df[df["Grupo"] == "Familia"].shape

In [ ]:
df[df["Grupo"] == "Establecimientos relevados"].shape

In [ ]:
# Elimino Familia y Establecimientos relevados
df = df[~df["Grupo"].isin(["Familia", "Establecimientos relevados"])].reset_index(drop=True)

In [ ]:
df["Grupo"].value_counts()

### Conclusión:
>
> Se decidió eliminar del análisis las categorías `Familia` y `Establecimientos relevados` dentro de la variable `Grupo`, ya que no representan grupos principales de productos comparables con el resto de las categorías.
>
> Luego de esta limpieza, el análisis se enfoca en los grupos más relevantes del dataset:
>
> `Alimentos y bebidas`, `Cuidado personal`, `Limpieza del hogar` y `Frutas y verduras`.

### Paso 5: Análisis de variables

#### Análisis de Variables Univariante Categóricas

Una **variable categórica** es un tipo de variable que puede tomar uno de un número limitado de categorías o grupos. Estos grupos suelen ser nominales, es decir, representan nombres o clases sin un orden numérico natural.

En este dataset, las variables categóricas permiten identificar dimensiones como el período relevado, el grupo de producto, el producto específico y el supermercado donde se registró el precio.

Antes de comenzar a graficar, debemos identificar cuáles son categóricas. En este caso, las variables categóricas son: `Periodo`, `Grupo`, `Producto` y `Super`.

La variable `Precio` no se considera categórica para el análisis, ya que representa un valor monetario y debería analizarse como variable numérica una vez convertida correctamente.

In [ ]:
df.dtypes

In [ ]:
#defino mis categorias con las que voy a trabajar en este analisis
variables_categoricas = ["Periodo", "Grupo", "Producto", "Super"]

df[variables_categoricas].nunique()


In [ ]:
#Producto lo trabajo por separado
variables_categoricas = ["Periodo", "Grupo", "Super"]

for col in variables_categoricas:
    plt.figure(figsize=(10, 4))
    df[col].value_counts().plot(kind="bar")
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=45, ha="right")
    plt.show()

In [ ]:
# Visual de producto
plt.figure(figsize=(10, 5))
df["Producto"].value_counts().head(15).plot(kind="bar")
plt.title("Top 15 productos más frecuentes")
plt.xlabel("Producto")
plt.ylabel("Frecuencia")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# Frecuencia por supermercado
df['Super'].value_counts()

In [ ]:
# Frecuencia por grupo
df['Grupo'].value_counts()

In [ ]:
# Frecuencia por periodo
df['Periodo'].value_counts().sort_index()

In [ ]:
# Precio promedio por supermercado
df.groupby('Super')['Precio'].agg(['count', 'mean', 'median', 'min', 'max']).sort_values('mean', ascending=False)

In [ ]:
# Precio promedio por grupo
df.groupby('Grupo')['Precio'].agg(['count', 'mean', 'median', 'min', 'max']).sort_values('mean', ascending=False)

In [ ]:
# Productos más caros en promedio
df.groupby('Producto')['Precio'].mean().sort_values(ascending=False).head(20)

In [ ]:
# Productos más baratos en promedio
df.groupby('Producto')['Precio'].mean().sort_values().head(20)

In [ ]:
# Histograma de precios
plt.figure(figsize=(10, 5))
plt.hist(df['Precio'], bins=50)
plt.title('Distribución de precios')
plt.xlabel('Precio')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
# Boxplot de precios por supermercado
plt.figure(figsize=(10, 5))
df.boxplot(column='Precio', by='Super', rot=45)
plt.title('Distribución de precios por supermercado')
plt.suptitle('')
plt.xlabel('Supermercado')
plt.ylabel('Precio')
plt.show()

In [ ]:
# Precio promedio por supermercado
precio_super = df.groupby('Super')['Precio'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
precio_super.plot(kind='bar')
plt.title('Precio promedio por supermercado')
plt.xlabel('Supermercado')
plt.ylabel('Precio promedio')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Precio promedio por grupo
precio_grupo = df.groupby('Grupo')['Precio'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
precio_grupo.plot(kind='bar')
plt.title('Precio promedio por grupo')
plt.xlabel('Grupo')
plt.ylabel('Precio promedio')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# Comparación supermercado vs grupo
tabla_super_grupo = pd.pivot_table(
    df,
    values='Precio',
    index='Grupo',
    columns='Super',
    aggfunc='mean'
)

tabla_super_grupo

In [ ]:
# Diferencia entre supermercados por producto
comparacion_productos = pd.pivot_table(
    df,
    values='Precio',
    index='Producto',
    columns='Super',
    aggfunc='mean'
)

comparacion_productos['diferencia_max_min'] = comparacion_productos.max(axis=1) - comparacion_productos.min(axis=1)

comparacion_productos.sort_values('diferencia_max_min', ascending=False).head(20)

#### Observaciones

A partir de los gráficos de las variables categóricas, se pueden extraer las siguientes conclusiones:

- **Periodo**: El dataset contiene registros para los 12 meses de 2025. La distribución es bastante pareja entre meses, aunque `Dic25`, `Feb25`, `Ene25` y `Abr25` son los períodos con mayor cantidad de registros. `Jul25` es el período con menor cantidad.
- **Grupo**: La categoría con mayor presencia es `Alimentos y bebidas`, con una diferencia importante frente al resto. Le siguen `Cuidado personal`, `Limpieza del hogar` y `Frutas y verduras`.
- **Grupo**: Antes de normalizar los textos, algunas categorías aparecían duplicadas por diferencias de mayúsculas y minúsculas, por ejemplo `Alimentos y bebidas` y `alimentos y bebidas`. Esto fue corregido al estandarizar el texto.
- **Super**: Existen registros para varios supermercados. Los que tienen mayor cantidad de observaciones son `Red Expres`, `Tienda Inglesa`, `Red Market`, `Disco`, `Devoto` y `Ta - Ta`.
- **Super**: También se observa una cantidad importante de registros sin supermercado informado, por lo que esta variable tenía valores nulos relevantes.
- **Producto**: Es la variable categórica con mayor variedad de valores. Esto es esperable, ya que el dataset contiene muchos productos distintos.
- **Producto**: Debido a la gran cantidad de productos, no conviene analizar todos en un único gráfico de barras. Es más claro observar solo los productos más frecuentes, por ejemplo el top 10 o top 15.

#### Análisis de Variables Univariante Numéricas

Una **variable numérica** es un tipo de variable que puede tomar valores numéricos, como enteros, decimales o valores monetarios. Normalmente se representa utilizando un histograma, para observar su distribución, y un gráfico de caja o boxplot, para identificar valores atípicos.

En este dataset, la principal variable numérica es `Precio`, ya que representa el valor monetario registrado para cada producto en cada supermercado y período.

Sin embargo, antes de graficarla es necesario convertirla a formato numérico, porque originalmente aparece como texto y utiliza coma decimal, por ejemplo `80,00`.

Por lo tanto, la variable numérica a analizar será: `Precio`.

In [ ]:
df["Precio"].describe()



In [ ]:
#Grafico

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["Precio"].hist(ax=axes[0], bins=30)
axes[0].set_title("Histograma de Precio")
axes[0].set_xlabel("Precio")
axes[0].set_ylabel("Frecuencia")

df.boxplot(column="Precio", ax=axes[1])
axes[1].set_title("Boxplot de Precio")
axes[1].set_ylabel("Precio")

plt.tight_layout()
plt.show()

#### Observaciones:
>
> - La combinación del histograma y el boxplot permite analizar la distribución de la variable `Precio` y detectar posibles valores atípicos.
> - La variable `Precio` presenta valores atípicos altos, ya que algunos registros tienen precios muy superiores al comportamiento general de los datos.
> - La distribución de `Precio` está sesgada hacia la derecha: la mayoría de los valores se concentran en precios bajos y medios, mientras que pocos registros presentan precios muy elevados.
> - La media de `Precio` es mayor que la mediana, lo que confirma que los valores altos influyen en el promedio.

### Análisis de Variables Multivariante

Tras analizar las variables una a una, es momento de estudiar cómo se relacionan entre sí. En este dataset no existe una variable objetivo o predictora como por ejemplo vimos en `Survived` en el caso del Titanic, por lo que el análisis multivariante se enfoca en entender cómo cambia la variable `Precio` según otras variables del dataset.

El objetivo principal es analizar si los precios varían según el `Grupo`, el `Producto`, el `Super` o el `Periodo`. Esto permite obtener conclusiones más claras sobre el comportamiento de los precios y tomar mejores decisiones sobre limpieza, eliminación de valores nulos u outliers.

Por ejemplo, la variable `Precio` presenta valores atípicos. Antes de eliminarlos, es importante revisar si estos outliers pertenecen a productos específicos, supermercados concretos o determinados grupos. Algunos precios altos podrían no ser errores, sino productos naturalmente más caros, como cremas, protectores solares o pañales.

Del mismo modo, la variable `Super` tiene una cantidad importante de valores nulos. Antes de eliminar esas filas, conviene analizar si esos registros tienen precios, productos o grupos relevantes, ya que podrían aportar información útil al análisis.

Por lo tanto, en este análisis multivariante se estudiará principalmente la relación entre `Precio` y las variables categóricas `Grupo`, `Producto`, `Super` y `Periodo`.

In [ ]:
df.groupby("Grupo")["Precio"].describe()
df.groupby("Super")["Precio"].describe()
df.groupby("Periodo")["Precio"].describe()

In [ ]:
#Para ver precios promedio por grupo:
df.groupby("Grupo")["Precio"].mean().sort_values(ascending=False)

In [ ]:
#Para ver precios promedio por supermercado:
df.groupby("Super")["Precio"].mean().sort_values(ascending=False)

In [ ]:
#Para ver precios promedio por mes:
df.groupby("Periodo")["Precio"].mean().sort_values(ascending=False)

#### Análisis numérico-numérico

Cuando las dos variables que se comparan tienen datos numéricos, se realiza un análisis numérico-numérico. Para comparar dos columnas numéricas se suelen utilizar diagramas de dispersión y análisis de correlaciones.

En este dataset, la única variable numérica relevante es `Precio`, ya que el resto de columnas (`Periodo`, `Grupo`, `Producto` y `Super`) son categóricas.

Por este motivo, no es posible realizar un análisis numérico-numérico tradicional entre dos variables del dataset. Tampoco tendría sentido calcular una matriz de correlación, ya que no existen dos o más variables numéricas para comparar entre sí.

En este caso, el análisis multivariante se enfocará principalmente en comparar la variable numérica `Precio` con variables categóricas como `Grupo`, `Super`, `Producto` y `Periodo`.


In [ ]:
#Comprobarlo con código:
df.select_dtypes(include="number").columns

In [ ]:
#Ver la correlación disponible:
df.select_dtypes(include="number").corr()
#Devuelve solo Precio, por eso no aporta mucho análisis.

#### Análisis categórico-categórico

Cuando las dos variables que se comparan son categóricas, se realiza un análisis categórico-categórico. Este tipo de análisis permite observar cómo se distribuyen las categorías de una variable dentro de otra.

En este dataset, las variables categóricas principales son `Periodo`, `Grupo`, `Producto` y `Super`. Por lo tanto, se pueden analizar relaciones como:

- qué grupos de productos aparecen en cada período,
- qué supermercados tienen mayor cantidad de registros por grupo,
- qué productos aparecen con mayor frecuencia en cada supermercado,
- y cómo se distribuyen los grupos de productos entre los supermercados.

Para este análisis se pueden utilizar tablas de frecuencia, tablas cruzadas y gráficos de barras agrupadas o apiladas.

In [ ]:
pd.crosstab(df["Grupo"], df["Super"])


In [ ]:
# Para verlo con porcentajes por fila:
pd.crosstab(df["Grupo"], df["Super"], normalize="index") * 100

In [ ]:
# Grafico:
tabla_grupo_super = pd.crosstab(df["Grupo"], df["Super"])

tabla_grupo_super.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

plt.title("Distribución de supermercados por grupo")
plt.xlabel("Grupo")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Super", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Periodo vs Grupo:
pd.crosstab(df["Periodo"], df["Grupo"])

In [ ]:
# Defino categorias
variables_categoricas = ["Periodo", "Grupo", "Producto", "Super"]

In [ ]:
# Grupo vs Super
#Sirve para ver qué supermercados tienen más registros por tipo de producto.

pd.crosstab(df["Grupo"], df["Super"])

In [ ]:
pd.crosstab(df["Grupo"], df["Super"], normalize="index") * 100

In [ ]:
# Periodo vs Grupo
# Sirve para ver si todos los grupos aparecen de forma pareja a lo largo de los meses.

pd.crosstab(df["Periodo"], df["Grupo"])

In [ ]:
pd.crosstab(df["Periodo"], df["Grupo"], normalize="index") * 100

In [ ]:
# Periodo vs Super
# Sirve para ver si todos los supermercados fueron relevados en todos los períodos.

pd.crosstab(df["Periodo"], df["Super"])

In [ ]:
# Super vs Producto
# Sirve para ver cuántos productos distintos tiene registrado cada supermercado.

df.groupby("Super")["Producto"].nunique().sort_values(ascending=False)

In [ ]:
# Grupo vs Producto
# Sirve para ver cuántos productos distintos hay dentro de cada grupo.

df.groupby("Grupo")["Producto"].nunique().sort_values(ascending=False)

In [ ]:
# Producto vs Super
# Sirve para ver en cuántos supermercados aparece cada producto.

df.groupby("Producto")["Super"].nunique().sort_values(ascending=False).head(20)

In [ ]:
# Combinaciones más frecuentes
# Sirve para detectar qué combinaciones de categorías aparecen más en el dataset.

df.groupby(["Grupo", "Super"]).size().sort_values(ascending=False).head(20)

In [ ]:
df.groupby(["Periodo", "Grupo"]).size().sort_values(ascending=False).head(20)

In [ ]:
df.groupby(["Producto", "Super"]).size().sort_values(ascending=False).head(20)

In [ ]:
tabla = pd.crosstab(df["Periodo"], df["Grupo"])

tabla.plot(kind="bar", stacked=True, figsize=(12, 6))

plt.title("Distribución de grupos por período")
plt.xlabel("Periodo")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=45)
plt.legend(title="Grupo", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
productos_por_super = df.groupby("Super")["Producto"].nunique().sort_values(ascending=False)

productos_por_super.plot(kind="bar", figsize=(12, 5))

plt.title("Cantidad de productos distintos por supermercado")
plt.xlabel("Supermercado")
plt.ylabel("Cantidad de productos distintos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("Productos distintos por grupo:")
print(df.groupby("Grupo")["Producto"].nunique().sort_values(ascending=False))

In [ ]:
print("Cantidad de registros por grupo:")
print(df["Grupo"].value_counts())

In [ ]:
print("Distribución de grupos por período:")
print(pd.crosstab(df["Periodo"], df["Grupo"]))

In [ ]:
print("Productos distintos por supermercado:")
print(df.groupby("Super")["Producto"].nunique().sort_values(ascending=False))

In [ ]:
print("Productos que aparecen en más supermercados:")
print(df.groupby("Producto")["Super"].nunique().sort_values(ascending=False).head(20))

In [ ]:
print("Combinaciones más frecuentes Grupo-Super:")
print(df.groupby(["Grupo", "Super"]).size().sort_values(ascending=False).head(20))

In [ ]:
print("Registros sin supermercado informado:")
print(df["Super"].isnull().sum())

##### Observaciones:

Del análisis categórico-categórico podemos obtener las siguientes conclusiones:

- `Alimentos y bebidas` es el grupo con mayor presencia en casi todos los cruces analizados. También es el grupo con mayor variedad de productos.
- `Cuidado personal` es el segundo grupo con mayor cantidad de registros y productos distintos.
- `Frutas y verduras` y `Limpieza del hogar` tienen una participación menor dentro del dataset.
- La distribución de grupos por período es bastante estable a lo largo del año. En todos los meses predominan los registros de `Alimentos y bebidas`, seguidos por `Cuidado personal`.
- Algunos períodos tienen menos registros que otros, especialmente `Jul25`, que presenta menor cantidad de observaciones en comparación con meses como `Dic25`, `Feb25` o `Ene25`.
- En cuanto a supermercados, `Tienda Inglesa`, `Red Market` y `Red Expres` son los que registran mayor variedad de productos distintos.
- Existen productos que aparecen en muchos supermercados, especialmente productos de cuidado personal como jabones y desodorantes. Esto indica que algunos productos están ampliamente distribuidos entre distintas cadenas.
- Las combinaciones más frecuentes se dan principalmente entre `Alimentos y bebidas` y supermercados como `Red Expres`, `Red Market`, `Tienda Inglesa`, `Ta - Ta`, `Disco` y `Devoto`.
- Luego de eliminar las categorías `Familia` y `Establecimientos relevados`, el análisis queda enfocado en los grupos principales de productos comparables.
-  Se observan **2.483 registros sin supermercado informado**, lo que puede afectar los análisis que comparan productos o grupos por cadena. Por este motivo, los resultados relacionados con `Super` deben interpretarse considerando esta ausencia de información.

##### Análisis de correlaciones

El análisis de correlaciones permite estudiar la relación entre variables numéricas. En un problema supervisado, normalmente se analiza la relación entre la variable objetivo y las variables predictoras. Sin embargo, en este dataset no existe una variable objetivo o target definida.

Además, el dataset cuenta con una única variable numérica principal: `Precio`. Las demás variables (`Periodo`, `Grupo`, `Producto` y `Super`) son categóricas.

Por este motivo, no es posible realizar un análisis de correlación tradicional entre múltiples variables numéricas. En su lugar, se puede estudiar cómo cambia el `Precio` según las categorías del dataset, por ejemplo comparando el precio promedio por `Grupo`, `Super`, `Producto` o `Periodo`.

Este análisis permite entender si ciertos grupos de productos, supermercados o períodos presentan precios más altos o más bajos.


In [ ]:
df.select_dtypes(include="number").corr()

In [ ]:
df.groupby("Grupo")["Precio"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("Super")["Precio"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("Periodo")["Precio"].mean().sort_values(ascending=False)

#### Análisis numérico-categórico

El análisis numérico-categórico permite estudiar cómo se comporta una variable numérica según las categorías de otra variable. En este dataset, la variable numérica principal es `Precio`, mientras que las variables categóricas son `Periodo`, `Grupo`, `Producto` y `Super`.

Este análisis permite responder preguntas como:

- qué grupos de productos tienen precios más altos,
- qué supermercados presentan precios promedio más elevados,
- cómo varían los precios según el período,
- y qué productos concentran los precios más altos.

Para este tipo de análisis no se calculan correlaciones tradicionales, ya que las variables categóricas no son numéricas. En su lugar, se utilizan agrupaciones, medidas estadísticas y gráficos como boxplots o barras comparativas.

In [ ]:
df.groupby("Grupo")["Precio"].describe()

In [ ]:
df.groupby("Super")["Precio"].describe()

In [ ]:
df.groupby("Periodo")["Precio"].describe()

In [ ]:
# Precio promedio por grupo:
df.groupby("Grupo")["Precio"].mean().sort_values(ascending=False)

In [ ]:
#Precio promedio por supermercado:
df.groupby("Super")["Precio"].mean().sort_values(ascending=False)

In [ ]:
#Precio promedio por período:
df.groupby("Periodo")["Precio"].mean().sort_values(ascending=False)

In [ ]:
# Boxplot de precio por grupo:

plt.figure(figsize=(10, 5))
df.boxplot(column="Precio", by="Grupo")
plt.title("Precio por grupo")
plt.suptitle("")
plt.xlabel("Grupo")
plt.ylabel("Precio")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
#Boxplot de precio por supermercado:

plt.figure(figsize=(12, 5))
df.boxplot(column="Precio", by="Super")
plt.title("Precio por supermercado")
plt.suptitle("")
plt.xlabel("Supermercado")
plt.ylabel("Precio")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# Estadísticas de Precio por Grupo
analisis_grupo = df.groupby("Grupo")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("promedio", ascending=False)

analisis_grupo

In [ ]:
# Estadísticas de Precio por Supermercado
analisis_super = df.groupby("Super")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("promedio", ascending=False)

analisis_super

In [ ]:
# Estadísticas de Precio por Periodo
analisis_periodo = df.groupby("Periodo")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("promedio", ascending=False)

analisis_periodo

In [ ]:
# Productos con precios más altos
productos_mas_caros = df.groupby("Producto")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("maximo", ascending=False)

productos_mas_caros.head(20)

In [ ]:
# Productos con mayor mediana de precio
productos_mayor_mediana = df.groupby("Producto")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("mediana", ascending=False)

productos_mayor_mediana.head(20)

##### Observaciones:

- Existe una relación clara entre `Grupo` y `Precio`. El grupo `Cuidado personal` presenta los precios más altos en promedio y mediana, especialmente por productos como protectores solares, pañales, cremas y tintas.
- `Alimentos y bebidas` es el grupo con mayor cantidad de registros, pero sus precios tienden a ser más bajos que los de `Cuidado personal`.
- `Limpieza del hogar` y `Frutas y verduras` presentan precios más bajos y una menor dispersión en comparación con los otros grupos.
- También se observan diferencias entre supermercados. Algunos como `San Roque`, `Pigalle`, `FarmaGlobal` y `Farmashop` muestran precios promedio más altos, lo cual puede estar relacionado con que venden más productos de cuidado personal o farmacia.
- El análisis por `Periodo` muestra que los precios promedio varían entre meses, aunque la diferencia no es tan fuerte como la observada entre grupos o supermercados.
- Los valores máximos de `Precio` afectan mucho los promedios, por lo que conviene comparar también la mediana. Esto permite reducir el efecto de los outliers.
- Al quitar outliers, las diferencias entre grupos y supermercados siguen existiendo, pero se vuelven menos extremas.
- Los productos con precios más altos pertenecen principalmente a categorías como pañales, protectores solares, tintas y cremas, por lo que no todos los outliers necesariamente son errores.

## Paso 6: Ingeniería de características

### Ingeniería de características

La **ingeniería de características** (*feature engineering*) es un proceso que implica crear nuevas variables a partir de las existentes, transformar datos o corregir formatos para mejorar el análisis y facilitar posibles modelos posteriores.

En los pasos previos ya se realizaron algunas tareas relacionadas con este proceso, como la eliminación de duplicados, el tratamiento de valores nulos, la normalización de textos y la conversión de la variable `Precio` a formato numérico.

En este dataset no existe una variable objetivo o *target* definida, por lo que la ingeniería de características se enfoca principalmente en mejorar la calidad de los datos y generar nuevas variables que permitan analizar mejor el comportamiento de los precios.

Algunas transformaciones útiles para este dataset son:

- convertir `Precio` a formato numérico,
- normalizar las variables categóricas (`Grupo`, `Producto`, `Super`),
- crear una variable de mes a partir de `Periodo`,
- crear una variable que indique si un precio es outlier,
- crear rangos de precio para clasificar productos en baratos, medios o caros,
- y calcular precios promedio por producto o supermercado.


In [ ]:
# Crear indicador de outlier:
Q1 = df.groupby("Producto")["Precio"].transform(lambda x: x.quantile(0.25))
Q3 = df.groupby("Producto")["Precio"].transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR

df["Limite_superior_producto"] = Q3 + 1.5 * IQR
df["Es_outlier_producto"] = df["Precio"] > df["Limite_superior_producto"]

df_outliers_producto = df[df["Es_outlier_producto"]].copy()

df_outliers_producto.sort_values("Precio", ascending=False)

In [ ]:
Q1 = df.groupby("Producto")["Precio"].transform(lambda x: x.quantile(0.25))
Q2 = df.groupby("Producto")["Precio"].transform(lambda x: x.quantile(0.50))
Q3 = df.groupby("Producto")["Precio"].transform(lambda x: x.quantile(0.75))

df["Rango_precio_producto"] = pd.Series(index=df.index, dtype="object")

df.loc[df["Precio"] <= Q1, "Rango_precio_producto"] = "Bajo"
df.loc[(df["Precio"] > Q1) & (df["Precio"] <= Q2), "Rango_precio_producto"] = "Medio bajo"
df.loc[(df["Precio"] > Q2) & (df["Precio"] <= Q3), "Rango_precio_producto"] = "Medio alto"
df.loc[df["Precio"] > Q3, "Rango_precio_producto"] = "Alto"

In [ ]:
df[["Producto", "Super", "Precio", "Rango_precio_producto"]].sort_values(
    ["Producto", "Precio"]
)

In [ ]:
# Crear mes desde `Periodo`:
mapa_meses = {
    "Ene25": 1,
    "Feb25": 2,
    "Mar25": 3,
    "Abr25": 4,
    "May25": 5,
    "Jun25": 6,
    "Jul25": 7,
    "Ago25": 8,
    "Sep25": 9,
    "Oct25": 10,
    "Nov25": 11,
    "Dic25": 12
}

df["Mes"] = df["Periodo"].map(mapa_meses)

In [ ]:
df["Periodo"].unique()

In [ ]:
df["Mes"].isnull().sum()

#### Análisis de outliers

**Intro to outliers**

Un **valor atípico** (*outlier*) es un dato que se aleja significativamente del comportamiento general del conjunto de datos.

En este dataset, el análisis de outliers se concentra principalmente en la variable `Precio`, ya que es la única variable numérica relevante. Los valores atípicos en precios pueden deberse a:

- errores de carga,
- diferencias reales entre tipos de productos,
- productos naturalmente más caros,
- promociones o precios especiales,
- o diferencias entre supermercados.

Para tratar los outliers existen varias estrategias:

- **Eliminarlos**: se eliminan las filas con precios atípicos. Esta opción puede ser útil si se confirma que son errores o si distorsionan mucho el análisis.
- **Mantenerlos**: se conservan porque pueden representar productos realmente caros o información importante.
- **Reemplazarlos**: se sustituyen por otro valor, como la mediana o un límite máximo, para reducir su efecto sin eliminar filas completas.

En este caso, antes de decidir qué hacer con los outliers, es importante analizarlos por `Grupo`, `Producto`, `Super` y `Periodo`.

**Outliers: Análisis descriptivo**

El análisis descriptivo permite caracterizar la variable `Precio` mediante medidas como la media, mediana, desviación estándar, mínimo, máximo y cuartiles.

La función `.describe()` de pandas permite obtener rápidamente estos indicadores y detectar si existen valores extremos. Si la media es mucho mayor que la mediana, esto puede indicar una distribución sesgada hacia precios altos.

In [ ]:
df["Precio"].describe()

In [ ]:
#Para ver media y mediana juntas:

print("Media:", df["Precio"].mean())
print("Mediana:", df["Precio"].median())
print("Máximo:", df["Precio"].max())

In [ ]:
# Ya formule IQR y tengo limite superior, creo inferior
limite_inferior = Q1 - 1.5 * IQR
limite_inferior, limite_superior

In [ ]:
# Para guardar los outliers:
df_outliers = df[
    (df["Precio"] < limite_inferior) |
    (df["Precio"] > limite_superior)
].copy()

df_outliers

In [ ]:
#Para resumirlos:
df_outliers.groupby("Grupo")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("cantidad", ascending=False)

In [ ]:
df_outliers.groupby("Producto")["Precio"].agg(
    cantidad="count",
    minimo="min",
    mediana="median",
    promedio="mean",
    maximo="max"
).sort_values("maximo", ascending=False).head(20)

In [ ]:
#Como el precio no es negativo, me enfoco en los superiores:
df_outliers = df[df["Precio"] > limite_superior].copy()

In [ ]:
df_outliers.sort_values("Precio", ascending=False).head(20)

In [ ]:
df.sort_values("Precio", ascending=False).head(10)

In [ ]:
df.describe().T

Si bien la experiencia es un componente importante en el análisis de los resultados de la tabla anterior, podemos utilizar ciertas reglas para detectar valores atípicos, como observar el valor mínimo y máximo de una variable y compararlo con sus percentiles 25%, 50% y 75%.

> Observaciones
>
> La columna `Precio` tiene una media aproximada de **260.88**, una mediana o percentil 50% de **131**, y un valor máximo de **25.800**.
>
> Esta diferencia indica que existen precios muy altos en comparación con la mayoría de los registros. Como la media es bastante mayor que la mediana, la distribución está sesgada hacia la derecha.
>
> El valor máximo de **25.800** podría parecer un valor atípico o incluso un posible error de carga. Sin embargo, también podría corresponder a un producto efectivamente caro, suena a que es muy caro para que lo sea, pero debemos de investigar.
>
> Por este motivo, antes de eliminarlo, decidimos analizar a qué `Producto`, `Grupo`, `Super` y `Periodo` pertenece ese registro.


In [ ]:
df_outliers_superiores = df[df["Precio"] > limite_superior].copy()

df_outliers_superiores.sort_values("Precio", ascending=False)

In [ ]:
df_outliers_superiores = df_outliers_superiores.sort_values(
    "Precio",
    ascending=False
).reset_index(drop=True)

df_outliers_superiores

**Outliers: Visualización**

Dibujar un diagrama de caja de la variable `Precio` permite identificar visualmente los valores atípicos. En este tipo de gráfico, los puntos que quedan alejados del cuerpo principal de la caja representan precios que se salen del rango esperado según la distribución de los datos.

En este dataset, el boxplot es especialmente útil porque `Precio` presenta valores muy altos en comparación con la mayoría de los registros. Estos valores pueden corresponder a errores de carga o a productos naturalmente más caros.


In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column="Precio")

plt.title("Boxplot de Precio")
plt.ylabel("Precio")
plt.show()

In [ ]:
#Para verlo por grupo:

plt.figure(figsize=(10, 5))
df.boxplot(column="Precio", by="Grupo")

plt.title("Boxplot de Precio por Grupo")
plt.suptitle("")
plt.xlabel("Grupo")
plt.ylabel("Precio")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
# visualización más clara sin que los extremos aplasten el gráfico:

plt.figure(figsize=(8, 5))
df.boxplot(column="Precio")
plt.ylim(0, 1000)

plt.title("Boxplot de Precio con límite visual")
plt.ylabel("Precio")
plt.show()

In [ ]:
# Guardo los outliers por producto para revisarlos si quiero
df_outliers_producto = df[df["Es_outlier_producto"]].copy()

In [ ]:
# Elimino los outliers
df = df[~df["Es_outlier_producto"]].reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
#Verifico eliminados
df_outliers_producto.shape[0]

In [ ]:
df

#### Observaciones y conclusiones

> En este dataset, la variable afectada por outliers es `Precio`, ya que es la única variable numérica relevante.
>
> En una primera revisión general, se observaron precios muy superiores al comportamiento habitual del dataset. Sin embargo, comparar todos los precios entre sí puede ser poco adecuado, porque cada producto tiene un rango de precios propio.
>
> Por ejemplo, un precio alto puede ser normal para productos como protectores solares, pañales, tintas o cremas, pero podría ser atípico para productos más baratos.
>
> Por este motivo, se decidió detectar los outliers de `Precio` comparando cada registro contra los precios de su mismo `Producto`.
>
> Para cada producto se calculó el límite superior utilizando el rango intercuartílico (`IQR`). Luego, se marcaron como outliers aquellos registros cuyo `Precio` superaba el límite superior correspondiente a ese producto.
>
> Finalmente, los registros identificados como outliers por producto fueron separados en `df_outliers_producto` y eliminados del dataframe principal.

#### Análisis de valores faltantes

### Análisis de valores faltantes

Un **valor faltante** (*missing value*) es un dato que no tiene valor asignado en una observación para una variable específica.

Este tipo de valores son comunes y pueden surgir por distintas razones:

- errores en la carga o recolección de datos,
- información no disponible al momento del relevamiento,
- productos sin precio informado,
- supermercados no identificados,
- o datos que no aplican para ciertos registros.

En este dataset, los valores faltantes pueden aparecer en variables como `Grupo`, `Producto`, `Super` y `Precio`. Para tratarlos existen varias estrategias:

- **Eliminarlos**: se eliminan las filas que contienen valores nulos. Es una opción simple, pero puede hacer que se pierda información útil.
- **Imputación numérica**: se utiliza para rellenar valores faltantes en variables numéricas. En este caso, podría aplicarse a `Precio`, reemplazando los nulos por la media, mediana o algún valor calculado por grupo o producto.
- **Imputación categórica**: se utiliza para rellenar valores faltantes en variables categóricas como `Grupo`, `Producto` o `Super`. Una opción habitual es reemplazar los nulos por la moda o por una categoría como `Sin dato`.

En este análisis, antes de decidir si eliminar o imputar, es importante revisar cuántos valores faltantes tiene cada columna y qué impacto tendría eliminarlos.

In [ ]:
df.shape[0]

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum() / df.shape[0]

In [ ]:
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df.isnull().any(axis=1).sum()

In [ ]:
df.columns

In [ ]:
# Elimino filas con valores nulos
df = df.dropna(subset=["Grupo", "Producto", "Super", "Precio"]).reset_index(drop=True)

In [ ]:
df[df.isnull().any(axis=1)]

##### Observaciones

> En este dataset se identificaron valores faltantes en variables como `Grupo`, `Producto`, `Super` y `Precio`.
>
> Estos valores faltantes pueden afectar el análisis, especialmente en los cruces por supermercado, producto o grupo, y también en los cálculos estadísticos de `Precio`.
>
> En lugar de imputar estos valores, se decidió eliminar las filas con datos faltantes, ya que el objetivo es trabajar únicamente con registros completos y evitar introducir valores artificiales.

##### Conclusiones

> Se aplicó `dropna()` para eliminar las observaciones incompletas.
>
> Esta decisión permite conservar solo registros con información completa en todas las columnas principales del dataset.
>
> Luego de esta limpieza, los análisis posteriores de precios, productos, grupos, supermercados y períodos se realizan sobre datos completos, reduciendo posibles distorsiones por información faltante.

#### Inferencia de nuevas características


Otro uso típico en esta ingeniería es la de la obtención de nuevas características mediante la "fusión" de dos o más ya existentes. De esta forma podemos simplificar el número de variables y trazar nuevas relaciones con el target.


In [ ]:
df["Rango_precio"] = pd.cut(
    df["Precio"],
    bins=[0, 100, 300, 600, df["Precio"].max()],
    labels=["Bajo", "Medio", "Alto", "Muy alto"]
)

In [ ]:
df["Rango_precio"].value_counts()

#### Observación
>
> En este dataset, la variable `Precio` representa el valor monetario de cada producto relevado. A partir de esta variable se puede crear una nueva característica llamada `Rango_precio`, que permita clasificar los productos en categorías como precio bajo, medio, alto o muy alto.
>
> Esta nueva variable facilita el análisis, ya que permite comparar grupos de productos, supermercados o períodos según rangos de precios, sin depender únicamente del valor exacto del precio.

In [ ]:
df["Rango_precio"].value_counts()

In [ ]:
df["Es_outlier_precio"] = df["Precio"] > df["Limite_superior_producto"]

In [ ]:
df["Es_outlier_precio"].value_counts()

#### Observación
>
> Además, dado que la variable `Precio` presenta valores atípicos, se puede crear una nueva característica llamada `Es_outlier_precio`. Esta variable indica si un precio supera el límite superior calculado mediante el rango intercuartílico.
>
> Esto permite conservar la información original, pero identificar fácilmente qué registros tienen precios atípicos para analizarlos por separado.

### Incorporo nuevas variables

In [ ]:
def extraer_tipo_envase(producto):
    producto = str(producto).lower()

    if "botella" in producto:
        return "Botella"
    elif "bidón" in producto or "bidon" in producto:
        return "Bidón"
    elif "paquete" in producto:
        return "Paquete"
    elif "envase" in producto:
        return "Envase"
    elif "lata" in producto:
        return "Lata"
    elif "sachet" in producto:
        return "Sachet"
    elif "caja" in producto:
        return "Caja"
    elif "bolsa" in producto:
        return "Bolsa"
    elif "tetrabrick" in producto or "tetra" in producto:
        return "Tetrabrick"
    elif "frasco" in producto:
        return "Frasco"
    elif "pote" in producto:
        return "Pote"
    elif "aerosol" in producto:
        return "Aerosol"
    elif "spray" in producto:
        return "Spray"
    elif "rollo" in producto or "rollos" in producto:
        return "Rollo"
    elif "unidad" in producto or "un." in producto or " us." in producto or " us" in producto:
        return "Unidad"
    elif (
        "kg" in producto
        or "gr" in producto
        or "grs" in producto
        or "ml" in producto
        or "cm3" in producto
        or "lts" in producto
        or "lt." in producto
    ):
        return "Al peso"
    else:
        return "Sin identificar"

df["Tipo_envase"] = df["Producto"].apply(extraer_tipo_envase)

In [ ]:
df["Tipo_envase"].value_counts()

In [ ]:
df.groupby("Tipo_envase")["Precio"].agg(
    cantidad="count",
    mediana="median",
    promedio="mean",
    minimo="min",
    maximo="max"
).sort_values("cantidad", ascending=False)

In [ ]:
df["Tipo_envase"].value_counts().plot(kind="bar", figsize=(10, 5))

plt.title("Cantidad de productos por tipo de envase")
plt.xlabel("Tipo de envase")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
df.boxplot(column="Precio", by="Tipo_envase")

plt.title("Distribución de precios por tipo de envase")
plt.suptitle("")
plt.xlabel("Tipo de envase")
plt.ylabel("Precio")
plt.xticks(rotation=45, ha="right")
plt.show()

Se creó la variable `Tipo_envase` a partir del texto de `Producto`, identificando palabras clave como botella, paquete, lata, caja, bolsa o aerosol. Esta nueva variable permite analizar si el tipo de presentación del producto se relaciona con diferencias de precio o frecuencia dentro del dataset.

### Incorporación de indicadores que creamos con DAX en proyecto previo

In [ ]:
# Setteo trimestre
df["Trimestre"] = pd.cut(
    df["Mes"],
    bins=[0, 3, 6, 9, 12],
    labels=["T1", "T2", "T3", "T4"]
)

In [ ]:
# Mapeo de semestre
df["Semestre"] = df["Mes"].apply(lambda x: "S1" if x <= 6 else "S2")

In [ ]:
# Precio promedio del producto
df["Precio_promedio_producto"] = df.groupby("Producto")["Precio"].transform("mean")

In [ ]:
# Dif contra el promedio del producto
df["Diferencia_vs_promedio_producto"] = df["Precio"] - df["Precio_promedio_producto"]

In [ ]:
# Porcentaje de diferencia contra el promedio del producto
df["Porcentaje_vs_promedio_producto"] = (
    df["Diferencia_vs_promedio_producto"] / df["Precio_promedio_producto"] * 100
)

In [ ]:
# Supermercado barato/caro para ese producto
df["Posicion_precio_producto"] = df["Porcentaje_vs_promedio_producto"].apply(
    lambda x: "Mas barato que promedio" if x < 0 else "Mas caro que promedio"
)

In [ ]:
#Precio mínimo del producto
df["Precio_min_producto"] = df.groupby("Producto")["Precio"].transform("min")

In [ ]:
#Diferencia contra el mínimo del producto
df["Diferencia_vs_min_producto"] = df["Precio"] - df["Precio_min_producto"]

In [ ]:
#Ranking de precio por producto
df["Ranking_precio_producto"] = df.groupby("Producto")["Precio"].rank(method="dense")

In [ ]:
#Cantidad de supermercados por producto
df["Cantidad_supers_producto"] = df.groupby("Producto")["Super"].transform("nunique")

In [ ]:
# Precio promedio por trimestre
df.groupby("Trimestre")["Precio"].mean().plot(kind="bar", figsize=(8, 4))

plt.title("Precio promedio por trimestre")
plt.xlabel("Trimestre")
plt.ylabel("Precio promedio")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Precio promedio por tipo de envase
df.groupby("Tipo_envase")["Precio"].mean().sort_values(ascending=False).plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Precio promedio por tipo de envase")
plt.xlabel("Tipo de envase")
plt.ylabel("Precio promedio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#Top supermercados más caros vs promedio del producto
df.groupby("Super")["Porcentaje_vs_promedio_producto"].mean().sort_values(ascending=False).plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Diferencia porcentual promedio vs precio promedio del producto")
plt.xlabel("Supermercado")
plt.ylabel("% vs promedio del producto")
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="black", linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
#Distribución de diferencias vs promedio
df["Porcentaje_vs_promedio_producto"].hist(bins=40, figsize=(8, 4))

plt.title("Distribución del porcentaje vs promedio del producto")
plt.xlabel("% diferencia vs promedio del producto")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
#Cantidad de supermercados por producto
df["Cantidad_supers_producto"].hist(bins=20, figsize=(8, 4))

plt.title("Distribución de cantidad de supermercados por producto")
plt.xlabel("Cantidad de supermercados")
plt.ylabel("Cantidad de productos/registros")
plt.show()

In [ ]:
#Top productos con mayor diferencia entre supermercados
diferencia_productos = df.groupby("Producto")["Precio"].agg(
    minimo="min",
    maximo="max"
)

diferencia_productos["diferencia"] = (
    diferencia_productos["maximo"] - diferencia_productos["minimo"]
)

diferencia_productos.sort_values("diferencia", ascending=False).head(15)["diferencia"].plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Top 15 productos con mayor diferencia de precio entre supermercados")
plt.xlabel("Producto")
plt.ylabel("Diferencia de precio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#### Observaciones sobre nuevas variables

A partir de las variables originales del dataset se crearon nuevas características para enriquecer el análisis exploratorio.

- `Tipo_envase` permite clasificar los productos según su presentación, como botella, paquete, lata, aerosol, unidad o al peso. Esta variable ayuda a analizar si ciertos formatos son más frecuentes o presentan precios más altos.
- `Trimestre` permite agrupar los meses en períodos más amplios, facilitando el análisis de posibles variaciones estacionales en los precios.
- `Precio_promedio_producto` calcula el precio promedio de cada producto en el dataset.
- `Diferencia_vs_promedio_producto` mide cuánto se aleja cada precio del promedio de su mismo producto.
- `Porcentaje_vs_promedio_producto` permite comparar precios de forma relativa, indicando si un registro está por encima o por debajo del promedio del producto.
- `Cantidad_supers_producto` indica en cuántos supermercados aparece cada producto, lo cual sirve como medida de disponibilidad.

#### Observaciones sobre los gráficos

- En el gráfico de precio promedio por trimestre se puede observar si los precios presentan variaciones a lo largo del año. Si las barras son similares, la evolución de precios es relativamente estable; si alguna destaca, puede indicar un aumento o baja en ese período.
- El gráfico de cantidad de registros por `Tipo_envase` permite identificar cuáles son las presentaciones más comunes dentro del dataset. Las categorías con mayor frecuencia representan los formatos más habituales de los productos relevados.
- El gráfico de precio promedio por `Tipo_envase` permite comparar si ciertos formatos tienen precios más altos. Es importante interpretar este resultado con cuidado, ya que el precio puede depender más del tipo de producto que del envase.
- El gráfico de diferencia porcentual promedio por supermercado muestra qué cadenas tienden a estar por encima o por debajo del precio promedio de cada producto. Valores positivos indican precios relativamente más altos, mientras que valores negativos indican precios relativamente más bajos.
- La distribución del `Porcentaje_vs_promedio_producto` permite observar si la mayoría de los precios se concentran cerca del promedio o si existen diferencias importantes entre supermercados para un mismo producto.
- La distribución de `Cantidad_supers_producto` muestra cuán extendidos están los productos entre supermercados. Productos presentes en muchas cadenas tienen mayor disponibilidad, mientras que productos con baja presencia pueden ser más específicos de ciertas cadenas.
- El gráfico de productos con mayor diferencia entre precio mínimo y máximo permite detectar productos donde existe una variación importante entre supermercados. Esto puede ser útil para identificar oportunidades de ahorro o posibles inconsistencias en los precios.

#### Conclusiones

Las nuevas variables creadas permiten pasar de un análisis descriptivo general a un análisis más comparativo.

En particular, comparar cada precio contra el promedio de su mismo producto es más informativo que comparar precios de productos distintos entre sí, ya que cada producto tiene una escala de precios propia.

Estas variables permiten responder preguntas más útiles para el objetivo del proyecto, como qué supermercados tienden a ser más caros o más baratos para los mismos productos, qué productos tienen mayor variación de precio y qué presentaciones son más frecuentes dentro del dataset.

## Paso 7: Split (dos métodos o enfoques)

[Doc: scikit-learn train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

[Doc: SelectFromModel](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html)
- fit()
- transform()
- get_support()

### Enfoque elegido: Realizar cambios antes de hacer el split de datos

En este enfoque:

- Primero se escala/codifica todo el dataset
- Luego se divide en conjuntos de entrenamiento y prueba.

Ventaja:

Garantiza que los datos de entrenamiento y prueba están procesados de la misma manera, ya que se utilizan los mismos parámetros de escalado y/o sistema de codificación en todo el dataset.

Desventaja:

Introduce información del conjunto de prueba en el conjunto de entrenamiento (porque los parámetros de se calculan usando todo el dataset). Esto puede llevar a una sobreestimación del rendimiento del modelo, ya que el modelo ha "visto" indirectamente la distribución de los datos de test.


### Realizamos el split

In [ ]:
df.shape

In [ ]:
df

In [ ]:
# Dropeo columnas de limite de outlier antes de procesar
df = df.drop(columns=["Es_outlier_producto", "Es_outlier_precio", "Limite_superior_producto"], errors="ignore")


In [ ]:
df.columns

In [ ]:
# Crear carpeta si no existe
processed_path = Path('../data/processed')
processed_path.mkdir(parents=True, exist_ok=True)

# Guardar como Excel
df.to_csv(processed_path / "cadenas_unificadas_2025_procesado.csv", index=False)

In [ ]:
X = df.drop(columns=["Precio"])
y = df["Precio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

columnas_numericas = X.select_dtypes(include="number").columns
columnas_categoricas = X.select_dtypes(include=["str", "category"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), columnas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
    ]
)

X_train_preprocesado = preprocessor.fit_transform(X_train)
X_test_preprocesado = preprocessor.transform(X_test)

### Aprendizajes

#### División entre entrenamiento y prueba

Antes de entrenar un modelo, es recomendable dividir los datos en conjuntos de entrenamiento y prueba.

- `X_train`: variables predictoras usadas para entrenar el modelo.
- `X_test`: variables predictoras usadas para evaluar el modelo.
- `y_train`: valores reales usados durante el entrenamiento.
- `y_test`: valores reales usados para evaluar el rendimiento.
- `test_size=0.2`: reserva el 20% de los datos para prueba y utiliza el 80% para entrenamiento.
- `random_state=42`: permite que la división sea reproducible.

#### Escalado y codificación

No todas las columnas se procesan igual:

- Las variables numéricas se pueden escalar con `StandardScaler`.
- Las variables categóricas deben codificarse, por ejemplo con `OneHotEncoder`.
- No se debe aplicar `StandardScaler` directamente sobre columnas de texto como `Periodo`, `Grupo`, `Producto` o `Super`.

#### Evitar data leakage

La forma correcta de transformar los datos es:
```python
fit_transform()  # solo en entrenamiento
transform()      # solo en prueba

## Paso 8: Scaling & Encoding

[Sklearn - documentación de preprocessing](https://scikit-learn.org/stable/api/sklearn.preprocessing.html)

El **escalado de valores** (*feature scaling*) es un paso crucial en el preprocesamiento de datos para muchos algoritmos de Machine Learning. Es una técnica que cambia el rango de los valores de los datos para que puedan ser comparables entre sí.

Algunas técnicas son:
- Normalización, que es el proceso de cambiar los valores para que tengan una media de 0 y una desviación estándar de 1.
- Mínimo-Máximo, que transforma los datos para que todos los valores estén entre 0 y 1.

A continuación detallaremos cómo podemos aplicar cada una de ellas, pero recordemos que depende mucho del modelo que vayamos a querer entrenar

#### Normalización

In [ ]:
df.select_dtypes(include="number").columns

In [ ]:
norm_scaler = StandardScaler()

num_variables = ["Precio", "Mes"]

norm_features = norm_scaler.fit_transform(df[num_variables])

df_norm = pd.DataFrame(
    norm_features,
    index=df.index,
    columns=num_variables
)

df_norm.head()

En el paso anterior ya se realizó el preprocesamiento correcto para modelado, utilizando `ColumnTransformer`.
>
> Las variables numéricas se escalaron con `StandardScaler` y las variables categóricas se codificaron con `OneHotEncoder`.
>
> Además, el preprocesamiento se ajustó únicamente sobre `X_train` mediante `fit_transform()` y luego se aplicó sobre `X_test` con `transform()`, evitando data leakage.
>
> Por ese motivo, no es necesario volver a normalizar todo el dataframe completo.

#### Escalado Mínimo-Máximo


In [ ]:
num_variables = X_train.select_dtypes(include="number").columns

min_max_scaler = MinMaxScaler()

X_train_minmax = X_train.copy()
X_test_minmax = X_test.copy()

X_train_minmax[num_variables] = min_max_scaler.fit_transform(X_train[num_variables])
X_test_minmax[num_variables] = min_max_scaler.transform(X_test[num_variables])

`MinMaxScaler` transforma las variables numéricas para que queden en un rango entre 0 y 1.
>
> A diferencia de `StandardScaler`, que centra los datos usando media y desviación estándar, `MinMaxScaler` conserva la forma de la distribución pero cambia la escala.
>
> Para evitar `data leakage`, el escalador debe ajustarse únicamente con `X_train` y luego aplicarse a `X_test`.

### Encoding - Codificación de variables categóricas


Las variables categóricas, que contienen valores discretos y no numéricos, deben transformarse en una forma que los algoritmos de Machine Learning puedan entender.

Existen varias técnicas de codificación, cada una con sus ventajas y desventajas, dependiendo del tipo de datos y del modelo utilizado.

#### Label Encoding

Asigna un valor entero único a cada categoría.

Adecuado para variables categóricas ordinales donde el orden tiene importancia.

In [ ]:
# Creo copias de los conjuntos de entrenamiento: X_train | X_test
X_train_cat_le = X_train.copy()
X_test_cat_le = X_test.copy()

#Defino columnas categoricas
columnas_categoricas = ["Periodo", "Grupo", "Producto", "Super"]

#Creo bucle de instancias
for col in columnas_categoricas:
    label_encoder = LabelEncoder()

    label_encoder.fit(X_train[col])

    X_train_cat_le[col + "_le"] = label_encoder.transform(X_train[col])
    X_test_cat_le[col + "_le"] = label_encoder.transform(X_test[col])

X_train_cat_le.head()

#### One-Hot Encoding

Asigna un valor entero único a cada categoría.

Crea una columna binaria (0 o 1) para cada categoría.

Adecuado para variables categóricas nominales donde no hay un orden implícito.

No es conveniente cuando hay muchas categorías.


In [ ]:
cat_variables = ["Periodo", "Grupo", "Producto", "Super"]

onehot_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

onehot_encoder.fit(X_train[cat_variables])

X_train_cat_ohe = onehot_encoder.transform(X_train[cat_variables])
X_train_cat_ohe = pd.DataFrame(
    X_train_cat_ohe,
    index=X_train.index,
    columns=onehot_encoder.get_feature_names_out(cat_variables)
)

X_test_cat_ohe = onehot_encoder.transform(X_test[cat_variables])
X_test_cat_ohe = pd.DataFrame(
    X_test_cat_ohe,
    index=X_test.index,
    columns=onehot_encoder.get_feature_names_out(cat_variables)
)

X_train_cat_ohe.head()

#### Ordinal Encoding

Similar a Label Encoding, pero permite especificar el orden de las categorías.

Adecuado para variables categóricas ordinales.

In [ ]:
df_example = df[["Rango_precio_producto"]].copy()

df_example.head(15)

In [ ]:
mapa_rango_precio = {
    "Bajo": 1,
    "Medio bajo": 2,
    "Medio alto": 3,
    "Alto": 4
}

df["Rango_precio_producto_ordinal"] = df["Rango_precio_producto"].map(mapa_rango_precio)

In [ ]:
df_example

In [ ]:
ordinal_encoder = OrdinalEncoder(
    categories=[["Bajo", "Medio bajo", "Medio alto", "Alto"]]
)

df["Rango_precio_producto_ordinal"] = ordinal_encoder.fit_transform(
    df[["Rango_precio_producto"]]
)

df[["Rango_precio_producto", "Rango_precio_producto_ordinal"]].head(15)

In [ ]:
# Para que comience en 1
df["Rango_precio_producto_ordinal"] = df["Rango_precio_producto_ordinal"] + 1

In [ ]:
df[["Rango_precio_producto", "Rango_precio_producto_ordinal"]].head(20)

### Guardando los scalers & encoders


In [ ]:
# Guardamos utilizando Pickle (que nos permite guardar objetos de Python)
Path("../models").mkdir(parents=True, exist_ok=True)

with open("../models/precios_2025_procesado.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

## Paso 9: Selección de características


La **selección de características** (*feature selection*) es un proceso que implica seleccionar las características (variables) más relevantes de nuestro conjunto de datos para usarlas en la construcción de un modelo de Machine Learning, desechando el resto.

Existen varias razones para incluirlo en nuestro análisis exploratorio:

1. Simplificar el modelo para que sea más fácil de entender e interpretar.
2. Reducir el tiempo de entrenamiento del modelo.
3. Evitar el sobre-ajuste al reducir la dimensionalidad del modelo y minimizar el ruido y las correlaciones innecesarias.
4. Mejorar el rendimiento del modelo al eliminar las características irrelevantes.

Además, existen diversas técnicas para la selección de características. Muchas de ellas se basan a su vez en modelos supervisados entrenados o de clustering y tienes más información.

La librería `sklearn` contiene gran parte de las mejores alternativas para llevarla a cabo. Una de las herramientas que más se utilizan para realizar procesos de selección de características rápidos y con buenos resultados es `SelectKBest`. Esta función selecciona las `k` mejores características de nuestro conjunto de datos basándose en una función de un test estadístico. Este test estadístico normalmente es un ANOVA o un Chi-Cuadrado:

In [ ]:
selection_model = SelectKBest(score_func=f_regression, k=5)

selection_model.fit(X_train_preprocesado, y_train)

X_train_sel = selection_model.transform(X_train_preprocesado)
X_test_sel = selection_model.transform(X_test_preprocesado)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

variables_seleccionadas = feature_names[selection_model.get_support()]

variables_seleccionadas

In [ ]:
X_train_sel = pd.DataFrame(
    X_train_sel.toarray(),
    columns=variables_seleccionadas,
    index=X_train.index
)

X_test_sel = pd.DataFrame(
    X_test_sel.toarray(),
    columns=variables_seleccionadas,
    index=X_test.index
)

X_train_sel.head()

La selección de características, al igual que el entrenamiento del modelo en general, debe realizarse solo sobre el conjunto de entrenamiento y no sobre la totalidad del dataset.

Si se realizara sobre todos los datos, se podría introducir un sesgo conocido como **contaminación de datos** (*data leakage*). Esto ocurre cuando información del conjunto de prueba se utiliza indirectamente para tomar decisiones durante el entrenamiento, lo que puede llevar a una estimación demasiado optimista del rendimiento del modelo.

Por eso, la mejor práctica es dividir primero los datos en conjuntos de entrenamiento y prueba. Luego, la selección de características se ajusta únicamente con los datos de entrenamiento y se aplica posteriormente tanto al conjunto de entrenamiento como al conjunto de prueba.

En este caso, como la variable objetivo es `Precio`, se utiliza una selección de características orientada a regresión (`f_regression`) en lugar de Chi cuadrado. Las características seleccionadas se obtienen a partir del conjunto de entrenamiento ya preprocesado, utilizando las variables generadas por el `ColumnTransformer`.

In [ ]:
variables_seleccionadas

Las características seleccionadas por el modelo fueron:

In [ ]:
variables_seleccionadas